# Run test.py logic on sample images and compare model accuracy

Run this notebook from **cnn-jigsaw-solver**. It runs the same pipeline as `test.py` on each image in `sample_test_images/` for both `final.pkl` and `improve_final.pkl`, then prints accuracy vs `sample_test_labels.json`.

In [1]:
import json
import os
import pickle

import cv2
import cupy as cp
import numpy as np

from src.cnn.model import JigsawCNN
from src.training.utils import assign_patches, load_state_dict

c:\Users\mmart\Documents\coursework-ACV\.venv\Lib\site-packages\cupy\_environment.py:275: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [ ]:
SAMPLE_DIR = "sample_test_images"
LABELS_PATH = "sample_test_labels.json"
MODELS = ["final.pkl", "improve_final.pkl"]

In [12]:
with open(LABELS_PATH, "r") as f:
    gt_labels = json.load(f)

image_files = sorted(
    f for f in os.listdir(SAMPLE_DIR)
    if f.lower().endswith(".png")
)
print(f"Found {len(image_files)} images, {len(gt_labels)} labels.")

Found 50 images, 50 labels.


In [13]:
def run_model_on_images(model_path):
    """Same logic as test.py: load model, run forward + assign_patches on each image. Returns list of (filename, assignment)."""
    with open(model_path, "rb") as f:
        state_dict = pickle.load(f)
    net = JigsawCNN(in_channels=1)
    load_state_dict(net, state_dict)
    results = []
    for filename in image_files:
        path = os.path.join(SAMPLE_DIR, filename)
        image = cv2.imread(path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image, (32, 32))
        image = image.astype(np.float32) / 255.0
        if image.ndim == 2:
            image = image[np.newaxis, np.newaxis, :, :]
        else:
            image = image[np.newaxis, :, :, :].transpose(0, 3, 1, 2)
        X = cp.asarray(image)
        out = net.forward(X)
        probs = cp.asnumpy(out[0])
        assignment = assign_patches(probs)
        results.append((filename, assignment))
    return results

In [14]:
def accuracy_for_results(results):
    """Compare predictions to gt_labels. Accuracy = fraction of correct (position, original_index) over all 16*N."""
    correct = total = 0
    for filename, pred in results:
        if filename not in gt_labels:
            continue
        gt = np.array(gt_labels[filename], dtype=np.int32)
        for i in range(16):
            total += 1
            if pred[i] == gt[i]:
                correct += 1
    return correct / total if total else 0.0

In [15]:
print("Running same logic as test.py on each image for both models.\n")
for model_path in MODELS:
    if not os.path.isfile(model_path):
        print(f"{model_path}: file not found, skipping.")
        continue
    results = run_model_on_images(model_path)
    acc = accuracy_for_results(results)
    print(f"{model_path} -> accuracy: {acc:.4f}")
print("\nDone.")

Running same logic as test.py on each image for both models.

final.pkl -> accuracy: 0.4775
improve_final.pkl -> accuracy: 0.4775


ModuleNotFoundError: No module named 'torch'